In [8]:
!pip3 install openpyxl

In [2]:
import datetime
import os
import time
import requests
import pandas as pd
from io import StringIO
from time import sleep
from selenium import webdriver
import logging
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.common.alert import Alert
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import StaleElementReferenceException, TimeoutException, UnexpectedAlertPresentException, NoSuchElementException, NoAlertPresentException
from concurrent.futures import ThreadPoolExecutor, as_completed


In [2]:
def web_driver(headless=True, download_dir="downloaded_addys"):
    download_dir = os.path.abspath(download_dir)
    #print(download_dir)
    if not os.path.exists(download_dir):
        os.makedirs(download_dir)

    options = webdriver.ChromeOptions()
    options.add_argument("--verbose")
    options.add_argument('--no-sandbox')
    if headless: options.add_argument('--headless')
    options.add_argument('--disable-dev-shm-usage')  # Overcome limited resource problems
    options.add_argument('--disable-gpu')  # GPU hardware acceleration is unnecessary in headless mode
    
    prefs = {
        "download.default_directory": download_dir,
        "savefile.default_directory": download_dir,
        "download.directory_upgrade": True
    }
    options.add_experimental_option("prefs", prefs)

    driver = webdriver.Chrome(options=options)
    
    return driver

In [14]:
driver = web_driver(headless=True)

# Set download location to new_data/{link.split('/')[-1].split('.')[0]}/

driver.get("https://www.taxdatahub.com/")
link = driver.find_element(By.XPATH, "//tbody/tr[2]/td[1]/span[1]/a[1]")
link.click()

time.sleep(2)  # Replace with WebDriverWait if needed for dynamic loading

# Get the current window handles and switch to the new tab
driver.switch_to.window(driver.window_handles[-1])

owner_input = WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.ID, 'search-box-input')))
owner_input.send_keys("Soni")
# Press button at //body[1]/div[1]/div[1]/div[3]/div[1]/div[1]/div[1]/a[1]/svg[1]/path[1]
owner_search_button = WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.XPATH, "//a[contains(text(),'Search')]")))
owner_search_button.click()
# Press Download button at //a[contains(text(),'Download Excel')]
download_button = WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.CLASS_NAME, "btn-download")))
#download_button = driver.find_element(By.CLASS_NAME, "btn-download")

download_button.click()

# Press Accept button at //*[@id="notice-modal-download"]/div/div/div[3]/button
accept_button = WebDriverWait(driver, 5).until(EC.element_to_be_clickable((By.XPATH, "//*[@id='notice-modal-download']/div/div/div[3]/button")))
accept_button.click()

time.sleep(5)

driver.quit()

In [ ]:
url = 'https://docs.google.com/spreadsheets/d/1yQ-LfAp9KOCq74p9xWGBWNt4fUvsMctUy9bLYCgvrJI/export?format=csv'
response = requests.get(url)
response.raise_for_status()
last_names = pd.read_csv(StringIO(response.text))

for ln in last_names['Full List']:
    print(ln)

Adajania
Adani
Ambani
Amin
Antala
Barot
Bhagat
Bharucha
Bhatt
Bhavasar
Bhavsar
Brahmbhatt
But
Chauhan
Chavda
Chokshi 
Choksi
Dalal
Dani
Darji
Daruwala
Dave
Desai
Dholakia
Divatia
Doshi
Faldu
Gadhavi
Gajjar
Ganatra
Gandhi
Goswami
Jain
Jani
Javeri
Jha
Jhala
Joshi
Kadakia
Kakadia
Kapadia
Khambatta
Khatri
Kotadiya
Kotecha
Kothari
Lakhani
Madhvani
Mahajan
Maru
Mehta
Mistry
Modi
Munim
Naik
Nayak
Odedara
Odedra
Ojha
Oza
Pambhar
Panchal
Pandya
Paramar
Parekh
Parikh
Parmar
Patel
Pathak
Poonawalla
Popat
Purohit
Rana
Rathod
Rathwa
Raval
Rawal
Savalia
Shah
Sharma
Sheth
Shroff
Shukla
Solanki
Soni
Tailor
Thakkar
Tripathi
Trivedi
Upadhyay
Vaghela
Vidhani
Virani
Vyas
Zalavadia
Zaveri


In [29]:
county_link_indexes = [1,2,3,4,5,6,7,8,9]

driver = web_driver(headless=False)
driver.get("https://www.taxdatahub.com/")


for county_link_index in county_link_indexes:
    link = driver.find_element(By.XPATH, f"//tbody/tr[{county_link_index}]/td[1]/span[1]/a[1]")
    link.click()

    # time.sleep(2)  # Replace with WebDriverWait if needed for dynamic loading
    driver.switch_to.window(driver.window_handles[-1])

    owner_input = WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.ID, 'search-box-input')))
    
    for index, ln in enumerate(last_names['Full List']):
        if (index != 0) and (index % 1 == 0):
            driver.close();
            driver.switch_to.window(driver.window_handles[0])
            link.click()
            driver.switch_to.window(driver.window_handles[-1])
            owner_input = WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.ID, 'search-box-input')))
        
        time.sleep(0.5)
        owner_input.clear()
        owner_input.send_keys(ln)

        owner_search_button = WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.XPATH, "//a[contains(text(),'Search')]")))
        owner_search_button.click()
        owner_search_button.click()
        owner_search_button.click()
        
        try:
            download_button = WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.XPATH, "//a[contains(text(),'Download Excel')]")))
            download_button.click()
            
            #If accept button does not appear, click again
            try:
                accept_button = WebDriverWait(driver, 0.1).until(EC.element_to_be_clickable((By.XPATH, "//button[contains(text(),'I Accept')]")))
                accept_button.click()
            except TimeoutException:
                download_button.click()

            accept_button = WebDriverWait(driver, 0.1).until(EC.element_to_be_clickable((By.XPATH, "//button[contains(text(),'I Accept')]")))
            accept_button.click()
            
        except TimeoutException:
            print(f"No data found for {ln}")
            continue

        time.sleep(3)

    driver.close();
    driver.switch_to.window(driver.window_handles[0])

driver.quit()

No data found for Antala


NoSuchWindowException: Message: no such window: target window already closed
from unknown error: web view not found
  (Session info: chrome=132.0.6834.160)
Stacktrace:
0   chromedriver                        0x0000000100aeb7a4 cxxbridge1$str$ptr + 2589716
1   chromedriver                        0x0000000100ae405c cxxbridge1$str$ptr + 2559180
2   chromedriver                        0x0000000100687f5c cxxbridge1$string$len + 88260
3   chromedriver                        0x000000010066387c chromedriver + 129148
4   chromedriver                        0x00000001006f391c cxxbridge1$string$len + 529028
5   chromedriver                        0x00000001006f92a4 cxxbridge1$string$len + 551948
6   chromedriver                        0x00000001006c0e40 cxxbridge1$string$len + 321448
7   chromedriver                        0x00000001006c1a88 cxxbridge1$string$len + 324592
8   chromedriver                        0x0000000100ab68a0 cxxbridge1$str$ptr + 2372880
9   chromedriver                      

In [49]:
def download_wait(folder_size, folder_path=None, timeout=5):
    start_time = time.time()
    if folder_path is None:
        folder_path = os.path.abspath("downloaded_addys")
    while True:
        # Check if folder size increased
        new_size = len(os.listdir(folder_path))
        if new_size > folder_size:
            return {"downloaded": True, "new_size": new_size, "time": time.time() - start_time}
        elif time.time() - start_time > timeout:
            return {"downloaded": False, "new_size": new_size, "time": time.time() - start_time}

        time.sleep(0.1)

In [114]:
def get_link_last_name_excel(county_link_index, ln):
    
    link = driver.find_element(By.XPATH, f"//tbody/tr[{county_link_index}]/td[1]/span[1]/a[1]")
    link.click()
    # time.sleep(2)  # Replace with WebDriverWait if needed for dynamic loading
    driver.switch_to.window(driver.window_handles[-1])

    owner_input = WebDriverWait(driver, 10).until(EC.presence_of_element_located((By.ID, 'search-box-input')))
    owner_input.send_keys(ln)
    owner_search_button = WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.XPATH, "//a[contains(text(),'Search')]")))
    time.sleep(0.01)
    owner_search_button.click()
    
    time.sleep(5)

    # Get result count at CSS_Selector: div:nth-child(25) div.search-results div.results-top-bar b.me-2.doc-total-label:nth-child(1) > span.doc-total
    result_count = driver.find_element(By.XPATH, "//body/div[1]/div[2]/div[2]/div[3]/b[1]/span[1]")
    print(f"Result Count: {result_count.get_attribute('innerHTML')}")
    if result_count.get_attribute('innerHTML') == "0":
        time.sleep(0.1)
        print(f"No data found for {ln}")
        driver.close();
        driver.switch_to.window(driver.window_handles[0])
        return

    try:
        # Press Download button at //a[contains(text(),'Download Excel')]
        download_button = WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.CLASS_NAME, "btn-download")))
        download_button.click()
        # Press Accept button at //button[contains(text(),'I Accept')]
        accept_button = WebDriverWait(driver, 5).until(EC.element_to_be_clickable((By.XPATH, "//button[contains(text(),'I Accept')]")))
        accept_button.click()
    except TimeoutException:
        print(f"No data found for {ln}")
        driver.quit()
        return
    
    folder_size = len(os.listdir("downloaded_addys"))
    download_response = download_wait(folder_size=folder_size)
    if not download_response["downloaded"]:
        # Try again
        try:
            # Press Download button at //a[contains(text(),'Download Excel')]
            download_button = WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.CLASS_NAME, "btn-download")))
            download_button.click()
            # Press Accept button at //button[contains(text(),'I Accept')]
            accept_button = WebDriverWait(driver, 5).until(EC.element_to_be_clickable((By.XPATH, "//button[contains(text(),'I Accept')]")))
            accept_button.click()
            download_wait(folder_size=folder_size)
        except TimeoutException:
            print(f"No data found for {ln}")
            driver.quit()
            return

    time.sleep(1)

    driver.close();
    driver.switch_to.window(driver.window_handles[0])

In [115]:
county_link_indexes = [2,3,4,5,6,7,8,9]
driver = web_driver(headless=False)
driver.get("https://www.taxdatahub.com/")

for county_link_index in county_link_indexes:
    last_name_search = last_names['Full List'].str.cat(sep=', ')
    
    get_link_last_name_excel(county_link_index, last_name_search)
    time.sleep(5)

Result Count: 0
No data found for Adajania, Adani, Ambani, Amin, Antala, Barot, Bhagat, Bharucha, Bhatt, Bhavasar, Bhavsar, Brahmbhatt, But, Chauhan, Chavda, Chokshi , Choksi, Dalal, Dani, Darji, Daruwala, Dave, Desai, Dholakia, Divatia, Doshi, Faldu, Gadhavi, Gajjar, Ganatra, Gandhi, Goswami, Jain, Jani, Javeri, Jha, Jhala, Joshi, Kadakia, Kakadia, Kapadia, Khambatta, Khatri, Kotadiya, Kotecha, Kothari, Lakhani, Madhvani, Mahajan, Maru, Mehta, Mistry, Modi, Munim, Naik, Nayak, Odedara, Odedra, Ojha, Oza, Pambhar, Panchal, Pandya, Paramar, Parekh, Parikh, Parmar, Patel, Pathak, Poonawalla, Popat, Purohit, Rana, Rathod, Rathwa, Raval, Rawal, Savalia, Shah, Sharma, Sheth, Shroff, Shukla, Solanki, Soni, Tailor, Thakkar, Tripathi, Trivedi, Upadhyay, Vaghela, Vidhani, Virani, Vyas, Zalavadia, Zaveri
Result Count: 0
No data found for Adajania, Adani, Ambani, Amin, Antala, Barot, Bhagat, Bharucha, Bhatt, Bhavasar, Bhavsar, Brahmbhatt, But, Chauhan, Chavda, Chokshi , Choksi, Dalal, Dani, Darji

NoSuchWindowException: Message: no such window: target window already closed
from unknown error: web view not found
  (Session info: chrome=132.0.6834.160)
Stacktrace:
0   chromedriver                        0x0000000100b4f7a4 cxxbridge1$str$ptr + 2589716
1   chromedriver                        0x0000000100b4805c cxxbridge1$str$ptr + 2559180
2   chromedriver                        0x00000001006ebf5c cxxbridge1$string$len + 88260
3   chromedriver                        0x00000001006c787c chromedriver + 129148
4   chromedriver                        0x000000010075791c cxxbridge1$string$len + 529028
5   chromedriver                        0x000000010076a800 cxxbridge1$string$len + 606568
6   chromedriver                        0x0000000100724e40 cxxbridge1$string$len + 321448
7   chromedriver                        0x0000000100725a88 cxxbridge1$string$len + 324592
8   chromedriver                        0x0000000100b1a8a0 cxxbridge1$str$ptr + 2372880
9   chromedriver                      

In [14]:
df_2 = pd.read_excel(f'public/files/Data_By_Towns_Index/Burlington/Burlington Township.xlsx', index_col=0,header=0)

In [10]:
tdf = pd.read_excel("downloaded_addys/Essex.xlsx",header=0)
tdf.head()

/Users/sunilpc/Desktop/VSCode/LB_Screener/venv/lib/python3.11/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


,PamsPin,OwnerName,OwnerStreet,OwnerCityState,OwnerZipCode,Block,Lot,Qual,PropertyLocation,PropertyClassCode,...,DeedPage,SalePrice,SaleAssessment,PropertyUseCode,SR1A_Code,EPL_Code,Facility,InitialFilingDate,FurtherFilingDate,ExemptStatuteNumber
0,0701_402_24,"Desai, Jignesh & Shah, Pranav",111 Newark Avenue,"Belleville, New Jersey",7109,402.0,24.0,NaN,111 Newark Avenue,4A,...,863.0,120000,44100,NaN,NaN,0.0,NaN,01/01/0001,01/01/0001,NaN
1,0701_403_28,"Verdugo,Dani P&Acosta,Indira I",93-95 Newark Avenue,"Belleville, Nj",7109,403.0,28.0,NaN,93-95 Newark Avenue,2,...,91707.0,365000,303600,NaN,NaN,0.0,NaN,01/01/0001,01/01/0001,NaN
2,0701_505_20,"Patel,Nelti",90 Sanford Avenue,"Belleville, Nj",7109,505.0,20.0,NaN,90 Sanford Avenue,2,...,2537.0,130000,311500,NaN,26.0,0.0,NaN,01/01/0001,01/01/0001,NaN
3,0701_701_7_C0016,"Lopez, Julio Enrique & Falcon, Dani","9 Montgomery St, U-A16",Belleville Nj,7109,701.0,7.0,C0016,9 Montgomery Street U-A16,2,...,41754.0,280000,176900,0.0,0.0,0.0,NaN,01/01/0001,01/01/0001,NaN
4,0701_701_7_C0019,"Pandya,Tara",5 Montgomery St.U-B1,"Belleville, Nj",7109,701.0,7.0,C0019,5 Montgomery Street U-B1,2,...,7676.0,150000,146600,NaN,NaN,0.0,NaN,01/01/0001,01/01/0001,NaN


In [15]:
links = pd.read_excel("links.xlsx",header=0)['County'].unique()
links

array(['Atlantic', 'Bergen', 'Burlington', 'Camden', 'Cape May',
       'Cumberland', 'Essex', 'Gloucester', 'Hudson', 'Hunterdon',
       'Mercer', 'Middlesex', 'Monmouth', 'Morris', 'Ocean', 'Passaic',
       'Salem', 'Somerset', 'Sussex', 'Union', 'Warren'], dtype=object)